# 01 · Pull and clean

The front of the pipeline, and the only place the data is touched. Every
model notebook downstream reads what this one writes, which is the point:
three model families cannot be compared if each cleans its own text.

Order of operations, and none of it is arbitrary:

1. **Pull** from the ERP and taxonomy tables, four levels wide.
2. **Clean** — strip the leading vendor code, fold misspellings, run the
   two-phase regex, expand abbreviations, filter to known tokens, attach
   the vendor marker. The order matters and is enforced by the cleaner
   object rather than by the order of cells here.
3. **Filter** rare classes and rare paths.
4. **Count** tokens and n-grams, so the vectorization choices in notebook
   02 are made from evidence rather than habit.
5. **Save** both the cleaned data and the cleaner itself.

That last step is the one worth being careful about. The cleaner is
pickled and travels with every model bundle, so scoring cleans exactly
the way training did. A scoring path that cleans differently does not
raise — it just sees mostly unknown words and quietly gets worse.

In [0]:
import sys, pathlib
# Repo root on the path, whether this runs from /notebooks locally or from a
# Databricks Repos checkout. Editable installs are avoided on purpose: the
# cluster does not keep them between restarts.
ROOT = pathlib.Path.cwd()
if not (ROOT / "nn_classifier").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

import nn_classifier as nnc
from nn_classifier import paths
print(paths.describe())


[paths] scratch: /local_disk0/tmp (cluster SSD): PermissionError -- using /tmp/nn_classifier
environment: Databricks
  data dir:   /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data
  output dir: /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs
  scratch:    /tmp/nn_classifier  (115 GB free)
  device:     cpu


In [0]:
from nn_classifier import cleaning, data
from nn_classifier.config import (
    DataConfig, CleaningConfig, FilterConfig, load_config, save_config)

RAW_FILE = "raw_data.txt"
CLEAN_FILE = "cleaned_data.txt"

data_cfg = DataConfig()
clean_cfg = CleaningConfig(
    strip_leading_code=True,
    normalize_misspellings=True,   # to_normalize.csv, before the regex pass
    expand_abbrevs=True,           # abbrevs_to_expand.csv, after it
    # OFF, and this is the change notebook 09 argued for.
    #
    # The dictionary filter kept only tokens NLTK recognises as
    # English words. Industrial descriptions are mostly not English
    # words -- TiAlN, CNMG432, 4FL, series numbers, sizes -- so it
    # was deleting the discriminative content and leaving stubs.
    # 09 found 'insert' alone as the entire cleaned description on
    # 7,634 rows spanning 61 level-4 categories, and 'ta ser' on
    # 1,417 more.
    #
    # What replaces it is statistical rather than lexical:
    # cfg.filters.min_token_count, set in notebook 02, keeps any
    # token appearing often enough to be evidence, dictionary or
    # not. Set this back to True to reproduce the earlier runs.
    filter_to_known_tokens=False,
    attach_vendor=False,           # switch on for the vendor-aware variant
)
# min_token_count is deliberately absent here. Token pruning does
# not happen in this notebook -- it happens in 03/04/05, which read
# cfg.filters.min_token_count out of the saved config. Setting it
# on this object would look like it worked and do nothing.
filter_cfg = FilterConfig(min_class_count=6, min_path_count=2)

print(data_cfg)
print(clean_cfg)

DataConfig(path='/Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data/cleaned_data.txt', file_format='csv', delimiter='|', text_col='FullDesc', id_col='ID', vendor_col='Vallen ID', level_columns=('Level 1 ID', 'Level 2 ID', 'Level 3 ID', 'Level 4 ID'), name_columns=('Level 1 Name', 'Level 2 Name', 'Level 3 Name', 'Level 4 Name'), drop_null_text=True, drop_null_labels=True)
CleaningConfig(strip_leading_code=True, uppercase=False, normalize_misspellings=True, expand_abbrevs=True, filter_to_known_tokens=False, keep_abbrev_keys=True, min_token_length=2, attach_vendor=False, vendor_token_prefix='VND_')


## 1 · Pull

On Databricks this runs against the live tables. The SQL lives in
`data.HIERARCHY_SQL` rather than only in this cell, so the schema the rest
of the package assumes is written down somewhere it can be diffed when the
source tables change.

Everything comes back as strings. Category IDs like `00123` lose their
leading zeros the moment anything guesses a numeric dtype, and the
resulting key mismatch is silent.

In [0]:
PULL = paths.on_databricks()   # set False to reuse the file already on disk

if PULL:
    print(data.HIERARCHY_SQL)
    raw = data.run_pull(spark)          # noqa: F821  (spark is injected)
    data.save_table(raw, RAW_FILE)
else:
    raw = pd.read_csv(paths.data_path(RAW_FILE), delimiter="|", dtype="str")

print(f"{len(raw):,} rows, {raw.shape[1]} columns")
raw.head()


with 
    base as (
        select s.id as `ID`, 
               s.erp_cono as `Cono`, 
               s.erp_product_number as `Vallen ID`, 
               replace(i.descrip, ';', ' ') as `FullDesc`,
               s.category_l4_id, 
               s.global_vendor_number as `Global Vendno`
        from prd_bronze.stibo_ftp.sku s 
        inner join prd_bronze.openedge.icsp i 
        on s.erp_product_number = i.prod
        where s.st_row_current=1 
            and s.category_l4_id is not null 
            and i.st_row_current=1 
            and i.cono in (1,55) 
            and i.descrip is not null
    ), 

    hierarchy as (
        SELECT
            l1.id   AS `Level 1 ID`,
            l1.name AS `Level 1 Name`,
            l2.id   AS `Level 2 ID`,
            l2.name AS `Level 2 Name`,
            l3.id   AS `Level 3 ID`,
            l3.name AS `Level 3 Name`,
            l4.id   AS `Level 4 ID`,
            l4.name AS `Level 4 Name`
        FROM prd_bronze.stibo_ftp.hierarchy l

,ID,Cono,Vallen ID,FullDesc,category_l4_id,Global Vendno,Level 1 ID,Level 1 Name,Level 2 ID,Level 2 Name,Level 3 ID,Level 3 Name,Level 4 ID,Level 4 Name
0,I8IGTHF0H3HT,1,00892502046,D0748 F BLADE 7-1/4X48TH FERROVS,L4-12810826,FREUD,L1-12810533,Tools & Equipment,L2-12810536,Power Tools,L3-12810584,Power Saw Blades & Accessories,L4-12810826,Circular Saw Blades
1,IF3JYMND0HXW,1,00892538579,50-101 ROUTER BIT,L4-12784351,FREUD,L1-12783695,Cutting Tools,L2-12783697,Holemaking,L3-12783722,Specialty Drill Bits,L4-12784351,Router Drill Bits
2,I8HHYVD9KP78,1,008925DS0912BW5,9IN DEMO DEMON F/METAL,L4-12810833,FREUD,L1-12810533,Tools & Equipment,L2-12810536,Power Tools,L3-12810584,Power Saw Blades & Accessories,L4-12810833,Reciprocating Saw Blades
3,I8HJI1MPX8DQ,1,01011981-34-34,81-34-34 MAGNIFIER 10X ILLUM CODDINGTON B&L,L4-12810909,BAUS,L1-12810533,Tools & Equipment,L2-12810537,Test Instruments,L3-13340590,Inspection Equipment,L4-12810909,Magnifiers
4,IACF1JY0BD2N,1,01058203068,BLOCK GABLE 03068 1/4 CA BLE,L4-13322418,1-10582,L1-12800755,Lifting & Rigging,L2-12800757,Chains & Ropes,L3-12800775,Ropes,L4-13322418,Ropes


In [0]:
# The hierarchy lookup is built here, from the raw pull, and saved with
# every model. Names are restored onto predictions from this table -- not
# from a live join, which would let a later rename upstream retroactively
# change what an old run appears to have said.
lookup = data.build_hierarchy_lookup(raw, data_cfg.level_columns, data_cfg.name_columns)
data.save_table(lookup, "hierarchy_lookup.csv", delimiter=",")
print(f"{len(lookup):,} distinct paths")
lookup.head()

[data] wrote 4,656 rows to /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data/hierarchy_lookup.csv
4,656 distinct paths


,Level 1 ID,Level 2 ID,Level 3 ID,Level 4 ID,Level 1 Name,Level 2 Name,Level 3 Name,Level 4 Name
0,L1-12810533,L2-12810536,L3-12810584,L4-12810826,Tools & Equipment,Power Tools,Power Saw Blades & Accessories,Circular Saw Blades
1,L1-12783695,L2-12783697,L3-12783722,L4-12784351,Cutting Tools,Holemaking,Specialty Drill Bits,Router Drill Bits
2,L1-12810533,L2-12810536,L3-12810584,L4-12810833,Tools & Equipment,Power Tools,Power Saw Blades & Accessories,Reciprocating Saw Blades
3,L1-12810533,L2-12810537,L3-13340590,L4-12810909,Tools & Equipment,Test Instruments,Inspection Equipment,Magnifiers
4,L1-12800755,L2-12800757,L3-12800775,L4-13322418,Lifting & Rigging,Chains & Ropes,Ropes,Ropes


## 2 · Clean

`build_cleaner` resolves every reference list once and stores it on the
object: the NLTK word list, the good tokens, the abbreviation and
normalisation maps. That makes the cleaner self-contained, so scoring
needs no NLTK download and cannot pick up a different word list than
training used.

In [0]:
cleaner = cleaning.cleaner_from_config(clean_cfg)
print(cleaner.describe())

[deps] nltk not installed (needed by the leading-code test in cleaning); installing now
DescriptionCleaner(lists=['color_abbrevs', 'colors', 'sizes', 'unit_abbrevs', 'unit_plurals', 'units'], vocabulary=234,377 words, strip_leading_code=True, uppercase=False, expand_abbrevs=True (2,244 pairs), normalize_misspellings=True (23,302 pairs), filter_to_known_tokens=False)


In [0]:
# A handful of rows through the cleaner before committing to the whole
# catalogue. If the output here still has units, sizes or stray digits in
# it, no amount of model tuning will make up for it.
sample = raw.sample(12, random_state=0)[[data_cfg.text_col]].copy()
sample["cleaned"] = cleaner.clean_series(sample[data_cfg.text_col])
sample.rename(columns={data_cfg.text_col: "original"})

,original,cleaned
443728,DIE OBLONG B 1/2X3/4IN,DIE OBLONG B
530895,BLPPTA3438 3/8IN PASS THRU ADAPTER,IN PASS THROUGH ADAPTER
291959,EXACTEK #7 HSCO JL 135 SP STRAW,7 HIGH-SPEED-COBALT JOBBER-LENGTH SP STRAW
372632,(DV)PANT CARGO FLEX TWILL BLK 42X34,CARGO FLEX TWILL
368323,CRIMP 78S FL62 90D 12FL62 12HOSE,CRIMP 78S FL62 90D 12FL62 12HOSE
207144,SUIT LG/XL W/HOOD 300 ZYTRON ELASTIC WRISTS,SUIT HOOD ZYTRON ELASTIC WRISTS
46127,C43317 5/16 682RH HS END MILL CTD,682RH HIGH-SPEED END MILL COATED
184178,33497 WIRE BRUSH 5X.020X 5/8-11 KNOTTED CUP-SINGL,WIRE BRUSH KNOTTED CUPSINGL
187966,101178 TAPE 3INX60YD DUCT PC 622 SIL,TAPE DUCT PC SIL
112635,106050 INSERT PUNCH SQ PP92Q F-.136 S-0,INSERT PUNCH SQUARE PP92Q F136 S0


In [0]:
cleaned = cleaning.clean_descriptions_with(
    raw,
    cleaner,
    column=data_cfg.text_col,
    vendor_col=data_cfg.vendor_col if clean_cfg.attach_vendor else None,
    vendor_prefix=clean_cfg.vendor_token_prefix,
    drop_blank=True,
)
print(f"{len(raw):,} -> {len(cleaned):,} rows "
      f"({(len(raw) - len(cleaned)) / len(raw):.2%} dropped as unreadable)")
cleaned[["FullDesc_orig", data_cfg.text_col]].head(10)

[clean] dropped 521 rows left blank by cleaning
604,136 -> 603,615 rows (0.09% dropped as unreadable)


,FullDesc_orig,FullDesc
0,D0748 F BLADE 7-1/4X48TH FERROVS,F BLADE TH FERROVS
1,50-101 ROUTER BIT,ROUTER BIT
2,9IN DEMO DEMON F/METAL,DEMO DEMON FMETAL
3,81-34-34 MAGNIFIER 10X ILLUM CODDINGTON B&L,MAGNIFIER ILLUMINATED CODDINGTON BL
4,BLOCK GABLE 03068 1/4 CA BLE,BLOCK GABLE BLE
5,66243506604 WHEEL 01 3 X 3/8 X 3/8 57A60-J8VB,WHEEL 57A60J8VB
6,00949-1 PLUMBERS BRUSH 1IN SS,PLUMBER BRUSH STAINLESS-STEEL
7,14610 BRUSH TUBE BRASS LONG HANDLE 3/4X15OAL,BRUSH TUBE BRASS LONG HANDLE OVERALL-LENGTH
8,BRUSH CONDENSER SS 7/8,BRUSH CONDENSER STAINLESS-STEEL
9,00134 4IN NRW CRMP WIRE WHEEL .0118 TLN-4,4IN NARROW CRIMPED WIRE WHEEL TLN4


## 3 · Filter

Two filters, doing different jobs. `min_class_count` removes classes too
thin to learn — and, more immediately, too thin to appear in both halves
of a stratified split. Six is what a 5-fold split needs with one row
spare. `min_path_count` removes paths seen once; joint-path scoring can
only emit a path it saw in training, so a path with a single example is a
category the model will hallucinate from one description.

In [0]:
levels = list(data_cfg.level_columns)

before = data.class_distribution(cleaned, levels)
cleaned = data.drop_null_rows(cleaned, data_cfg.text_col, levels)
cleaned = cleaning.drop_rare_classes(cleaned, levels[-1], filter_cfg.min_class_count)
cleaned = cleaning.drop_rare_paths(cleaned, levels, filter_cfg.min_path_count)
after = data.class_distribution(cleaned, levels)

before.merge(after, on="level", suffixes=("_before", "_after"))

[tokens] dropped 1,163 of 4,655 classes at 'Level 4 ID' seen < 6 times (3,009 rows, 0.50%)


,level,n_classes_before,min_rows_before,median_rows_before,max_rows_before,classes_under_10_before,n_classes_after,min_rows_after,median_rows_after,max_rows_after,classes_under_10_after
0,Level 1 ID,23,104,10706.0,127157,0,23,100,10630.0,126723,0
1,Level 2 ID,158,3,608.0,104180,4,154,6,637.0,104010,3
2,Level 3 ID,791,1,125.0,33936,100,715,6,158.0,33928,44
3,Level 4 ID,4655,1,19.0,17775,1635,3492,6,35.0,17775,472


## 4 · Count

The answer to *what should `min_df` be*, measured rather than guessed.

Read `n_classes` in the n-gram table before `doc_count`. A token in 5,000
descriptions spread over 400 categories is a stopword in everything but
name; one in 30 descriptions all in the same category is the kind of
feature a classifier is built on. Document count alone cannot tell them
apart.

In [0]:
unigrams = cleaning.count_ngrams(
    cleaned[data_cfg.text_col], ngram_range=(1, 1), labels=cleaned[levels[-1]]
)
bigrams = cleaning.count_ngrams(
    cleaned[data_cfg.text_col], ngram_range=(2, 2), labels=cleaned[levels[-1]]
)
print(f"{len(unigrams):,} distinct unigrams, {len(bigrams):,} distinct bigrams")
unigrams.head(20)

113,972 distinct unigrams, 449,110 distinct bigrams


,ngram,n,doc_count,n_classes
0,drill,1,36896,227
1,steel,1,33550,1083
2,high,1,27097,695
3,glove,1,24521,61
4,speed,1,24502,358
5,socket,1,22252,224
6,bit,1,20330,196
7,insert,1,17248,318
8,dr,1,17082,302
9,hex,1,15604,265


In [0]:
print("unigrams")
display(cleaning.token_report(cleaned[data_cfg.text_col], (1, 1)))
print("unigrams + bigrams")
display(cleaning.token_report(cleaned[data_cfg.text_col], (1, 2)))

unigrams


min_doc_count,n_ngrams,share_of_total
1,113972,1.0
2,40566,0.3559295265503808
3,28075,0.2463324325272874
5,19013,0.16682167549924543
10,11432,0.10030533815323062


unigrams + bigrams


min_doc_count,n_ngrams,share_of_total
1,563082,1.0
2,193863,0.3442891088686905
3,125091,0.22215414451181179
5,75827,0.13466422297285296
10,38493,0.06836126887380524


In [0]:
# The tokens worth the most: frequent enough to learn from, concentrated
# enough in a few categories to be worth learning.
informative = unigrams[(unigrams["doc_count"] >= 20) & (unigrams["n_classes"] <= 5)]
print(f"{len(informative):,} highly discriminative unigrams")
informative.head(25)

1,938 highly discriminative unigrams


,ngram,n,doc_count,n_classes
659,m226,1,664,1
661,hsshsco,1,661,5
667,gt100,1,656,5
673,cl88,1,649,5
718,nitrided,1,613,2
788,hoodie,1,557,5
805,ht800wp,1,544,3
807,facet,1,542,3
810,zcarb,1,535,5
811,cl109,1,535,2


## 5 · Save

In [0]:
import joblib

data.save_table(cleaned, CLEAN_FILE)  # -> data/cleaned_data.txt
joblib.dump(cleaner, paths.output_path("cleaner.joblib"), compress=3)
unigrams.to_csv(paths.output_path("token_counts_unigram.csv"), index=False)
bigrams.to_csv(paths.output_path("token_counts_bigram.csv"), index=False)
# The front-half settings, written into the shared config so
# notebook 02 inherits them instead of starting from dataclass
# defaults. Without this the config saved with every model records
# filter_to_known_tokens=True regardless of what actually ran --
# scoring stays correct, because it uses the pickled cleaner above,
# but the recorded provenance is a lie.
#
# Load-then-overwrite, not a fresh RunConfig: 02 owns the
# vectorizer, vocabulary and Word2Vec settings, and clobbering them
# from here would silently reset decisions made downstream.
cfg = load_config()
cfg.data, cfg.cleaning = data_cfg, clean_cfg

# Field by field, not `cfg.filters = filter_cfg`. This notebook owns
# the ROW filters; notebook 02 owns the TOKEN filters. Assigning the
# whole object would reset min_token_count to its default of 1 every
# time 01 is re-run after 02 -- and 03 would then prune nothing while
# printing no warning about it. Exactly the failure this config
# round-trip exists to prevent.
cfg.filters.min_class_count = filter_cfg.min_class_count
cfg.filters.min_path_count = filter_cfg.min_path_count

save_config(cfg)
print(f"[config] token pruning stays at min_token_count="
      f"{cfg.filters.min_token_count} (notebook 02 owns it)")
print("done")

[data] wrote 600,606 rows to /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data/cleaned_data.txt
[config] no saved config at /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs/run_config.joblib; using defaults
[config] wrote /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs/run_config.joblib and /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs/run_config.json
[config] token pruning stays at min_token_count=1 (notebook 02 owns it)
done
